# Classical tensor-network model: minimal training run

`configs/tn.yaml`: each word is a tensor (MPS-decomposed, bond dimension 32, wires of dimension 16) contracted along the
caption's grammar into a 512-dimensional vector, scored against the frozen CLIP vector of the image (InfoNCE).
Needs the processed data (`python scripts/retrieve.py -cfg configs/tn.yaml`).

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')   # paths in the configs are relative to the repository root

from modules.config import setup_exp
from modules.factory import load_experiment
from modules.models.fusion.trainer import ContrastiveTrainer
from modules.models.fusion.evaluator import MMEvaluator
import torch

config, device, seed = setup_exp('configs/tn.yaml', seed=0, overrides=['epochs=3'])
exp = load_experiment(config, device)
print(f"{len(exp.train)} training and {len(exp.val)} validation captions on {device}")

In [ ]:
optimizer = torch.optim.Adam(list(exp.text_model.parameters()) + list(exp.image_model.parameters()), lr=config['lr'])
trainer = ContrastiveTrainer(exp.image_model, exp.text_model, optimizer, exp.loss_fn, device)
evaluator = MMEvaluator(exp.image_model, exp.text_model, device)
train_loader = exp.data.get_loader(exp.train, split='train')
val_loader = exp.data.get_loader(exp.val, split='val')

for epoch in range(config['epochs']):
    loss = trainer.train_epoch(train_loader, exp.data.eval_mapper)
    metrics = evaluator.eval_set(val_loader, config['dataset']['val']['diagnostics'], exp.data.eval_mapper)
    print(f"epoch {epoch} | loss {loss:.4f} | val hard_neg_acc {metrics['hard_neg_acc']:.4f}")

In [ ]:
results, _ = evaluator.benchmark(exp.data, config['dataset']['test'])
{k: round(v, 4) for k, v in results.items() if k.endswith('hard_neg_acc')}